# 02-State 深入：数据怎么流转

### 回顾：State 是什么

上一课我们学了：

> **外卖订单类比**  
> State 就像一张**外卖订单**，在流水线上流转。每个工位（节点）都能看到订单内容，然后修改自己负责的部分。

代码里，State 是一个 TypedDict：

In [1]:
from typing import TypedDict


class OrderState(TypedDict):
    customer: str
    dish: str
    status: str

节点函数返回要修改的字段，LangGraph 自动合并：

In [2]:
def take_order(state: OrderState):
    return {"status": "已接单"}  # 只改 status，其他不变

今天我们要深入：**当多个节点同时修改同一个字段时，会发生什么？**

* * *

### 一、一个会出错的场景

假设我们做一个**客服系统**，需要同时做两件事：

-   查订单状态
-   查用户历史

这两个任务**互不依赖**，可以并行执行：

<img src="../assets/02_parallel_tasks.png" width="500" />

两个节点都写 `result` 字段 → 冲突！

两个节点都要往 `result` 字段写数据：

In [3]:
from typing import TypedDict


class State(TypedDict):
    query: str
    result: str


def check_order(state):
    return {"result": "订单已发货"}


def check_history(state):
    return {"result": "用户是 VIP"}

> ⚠️ **会出错！**  
> 两个节点在**同一个超步（superstep）**里同时写 `result`，LangGraph 不知道该用哪个值。  
> 会报错：`InvalidUpdateError: At key 'result': Can receive only one value per step.`

这就是今天要解决的核心问题：**多个节点写同一个字段，怎么办？**

> 💡 **为什么串行不报错？** 如果「查订单 → 查历史」是先后执行的，每一步只有一个节点写 `result`，新值覆盖旧值，不会冲突。只有**并行**时同一步出现多个写入，才必须告诉 LangGraph 合并规则。

* * *

### 二、Reducer：解决冲突的规则

答案是：**Reducer**（归约器）。

Reducer 就是一个函数，告诉 LangGraph：「当同一个字段有多个新值时，怎么合并它们？」

> **类比：快递站分拣**  
> 想象一个快递站，同一个人可能收到多个包裹。
> 
> -   **默认行为**：只保留最新一个包裹，旧的扔掉（覆盖）
> -   **Reducer 行为**：把所有包裹都堆在一起（追加）

怎么指定 Reducer？用 `Annotated`：

In [4]:
from typing import Annotated
from typing import TypedDict


class State2(TypedDict):
    # 普通字段：新值覆盖旧值
    name: str
    # 带 Reducer 的字段：新值追加到列表
    items: Annotated[list, "some_reducer"]

`Annotated[类型, 元信息]` 的意思是「在类型上附加额外信息」。LangGraph 约定：附加的第二个参数就是这个字段的 Reducer 函数。

Reducer 本质上就是一个两参数函数：`reducer(旧值, 新值) → 合并后的值`。所以你完全可以**写自己的 Reducer**：

In [5]:
def dedupe_append(old: list, new: list) -> list:
    # 追加新值，但跳过已存在的
    return old + [x for x in new if x not in old]


class State3(TypedDict):
    tags: Annotated[list, dedupe_append]  # 自定义合并规则

**三种写法对比**

| 写法  | 行为  | 适用场景 |
| --- | --- | --- |
| name: str | 新值覆盖旧值 | 状态标记、单值字段 |
| items: Annotated\[list, add\] | 新值追加到列表 | 日志、并行结果收集 |
| messages: Annotated\[list, add\_messages\] | 智能合并消息 | 对话历史 |

* * *

### 三、operator.add：最简单的 Reducer

`operator.add` 对列表来说就是 `旧列表 + 新列表`，作用：**把新列表追加到旧列表后面**。

In [6]:
from operator import add


class State4(TypedDict):
    results: Annotated[list, add]  # 用 add 作为 Reducer

执行过程：

```text
graph TD
    A[初始状态<br/>results: []] --> B[节点A 输出<br/>results: 订单已发货]
    A --> C[节点B 输出<br/>results: 用户是VIP]
    B --> D[add 合并]
    C --> D
    D --> E[最终 State<br/>results: 订单已发货, 用户是VIP]
    
    style A fill:#f1f5f9,stroke:#cbd5e1
    style B fill:#eff6ff,stroke:#93c5fd
    style C fill:#f0fdf4,stroke:#86efac
    style D fill:#fef3c7,stroke:#fbbf24
    style E fill:#f0fdf4,stroke:#22c55e
```

两个节点的输出都保留了！

**解决并行问题**

回到刚才的客服系统，加上 Reducer 就不会报错了：

In [7]:
class State5(TypedDict):
    query: str
    results: Annotated[list, add]  # 改成 list + add


def check_order2(state):
    return {"results": ["订单已发货"]}  # 注意：返回的是列表！


def check_history2(state):
    return {"results": ["用户是VIP"]}

最终 State：`{"query": "...", "results": ["订单已发货", "用户是VIP"]}`

> ⚠️ **新手最容易踩的坑：** 用了 `add` 之后，节点必须返回**列表**（哪怕只有一个元素也要写成 `["xx"]`）。如果返回字符串 `"xx"`，`add` 会把字符串当字符序列拼接，结果变成一堆碎字符。

> **检查理解 ①**
> 
> 以下 State 定义中，`logs` 字段的行为是什么？
> 
> ```python
> class State(TypedDict):
>     logs: Annotated[list, add]
> ```
> 
> A. 每次写入会覆盖之前的 logs  
> B. 每次写入会追加到 logs 列表末尾  
> C. 每次写入会插入到 logs 列表开头
> 
> 答案：**B**。`Annotated[list, add]` 的作用就是追加。每次新值都会追加到列表末尾。

* * *

### 四、add_messages：消息专用 Reducer

做 AI 应用，最常见的是**对话历史**。LangGraph 提供了一个专门的消息 Reducer：`add_messages`。

In [8]:
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages


class State6(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

`add_messages` 比 `operator.add` 更智能：

| 功能  | operator.add | add_messages |
| --- | --- | --- |
| 追加新消息 | ✅   | ✅   |
| 按 ID 更新消息 | ❌   | ✅   |
| 删除消息（RemoveMessage） | ❌   | ✅   |
| 自动反序列化（dict → 消息对象） | ❌   | ✅   |

> **类比：聊天记录本**
> 
> -   `operator.add`：每次有人说话，就贴一张便利贴。旧的不撕，新的追加。
> -   `add_messages`：智能记录本。如果同一句话改了（ID 相同），会替换旧的便利贴。还能按 ID 删除。

**三种能力各看一个例子**

```python
from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage

# ① 追加：返回新消息，自动加到末尾
return {"messages": [AIMessage(content="你好！")]}

# ② 更新：id 相同 → 替换旧消息，而不是追加
return {"messages": [AIMessage(content="修正后的回答", id="msg-42")]}

# ③ 删除：返回 RemoveMessage → 按 id 删掉对应消息（常用于裁剪历史）
return {"messages": [RemoveMessage(id=m.id) for m in state["messages"][:-2]]}
```

第 ③ 个例子很实用：对话太长时，**只保留最近 2 条**，其余全删，防止上下文爆炸。

**什么时候用哪个？**

-   **简单并行结果收集** → `operator.add`
-   **对话历史、需要修改/删除** → `add_messages`

> 💡 **关键区别：** `add_messages` 通过消息的 `id` 字段判断是追加还是更新。如果两条消息 id 相同，它会替换而不是重复添加。

* * *

### 五、MessagesState：快捷方式

因为对话场景太常见了，LangGraph 提供了一个内置的 State 类型：

```python
from langgraph.graph import MessagesState
```

它等价于：

```python
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
```

直接用就行：

```python
from langgraph.graph import StateGraph, MessagesState

builder = StateGraph(MessagesState)
```

**扩展 MessagesState**

如果除了消息，还需要其他字段（比如用户 ID、文档列表），可以继承：

```python
from langgraph.graph import MessagesState

class MyState(MessagesState):
    user_id: str           # 自定义字段
    documents: list[str]   # 检索到的文档
```

这样既有 `messages`（带 Reducer），又有自己的字段（默认覆盖）。

> **继承的秘密**  
> `MyState` 继承 `MessagesState` 后，自动拥有 `messages` 字段，**不需要重新定义**。等价于：

```python
class MyState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]  # ← 继承来的，自动带 Reducer
    user_id: str                                         # ← 你加的，默认覆盖
    documents: list[str]                                 # ← 你加的，默认覆盖
```

节点函数里直接用：

```python
def 处理消息(state: MyState):
    # 读对话历史
    for msg in state["messages"]:
        print(msg.content)
    # 返回新消息（自动追加/合并到 messages）
    return {"messages": [AIMessage(content="你好！")]}
```

> **检查理解 ②**
> 
> 下面哪种方式可以正确定义一个带对话历史的 State？
> 
> A. `class State(TypedDict): messages: list`  
> B. `class State(TypedDict): messages: Annotated[list, add_messages]`  
> C. `class State(MessagesState): messages: list`
> 
> 答案：**B**。A 没有 Reducer 会覆盖，C 不能重新定义 messages 字段。

* * *

### 六、多 Schema：输入输出分离

有时候，你不想暴露所有 State 字段给用户。比如：

-   用户只需要输入一个问题
-   内部处理时需要很多中间变量
-   最终只返回一个答案

可以用**多个 Schema**：

In [9]:
class InputState(TypedDict):
    question: str  # 用户输入


class OutputState(TypedDict):
    answer: str  # 最终输出


class OverallState(TypedDict):
    question: str  # 用户输入
    answer: str  # 最终输出
    context: str  # 内部检索的上下文（不暴露给用户）
    reasoning: str  # 推理过程（不暴露给用户）

创建图时指定输入输出 Schema：

```python
builder = StateGraph(
    OverallState,
    input_schema=InputState,
    output_schema=OutputState
)
```

> 💡 **版本提示：** `input_schema` / `output_schema` 是 langgraph 0.6+ 的参数名。如果你看到旧教程用 `input=` / `output=`，那是已废弃的旧写法。

```text
graph LR
    subgraph 用户视角
        A[InputState<br/>question: str] --> B[图]
        B --> C[OutputState<br/>answer: str]
    end
    subgraph 内部实际
        D[OverallState<br/>question + answer + context + reasoning]
    end
    
    style A fill:#eff6ff,stroke:#93c5fd
    style B fill:#667eea,stroke:#764ba2,color:#fff
    style C fill:#f0fdf4,stroke:#86efac
    style D fill:#f8fafc,stroke:#cbd5e1
```

**好处：**

-   用户不需要知道内部实现细节
-   输出干净，只有需要的字段
-   内部可以自由添加中间变量

> **检查理解 ③**
> 
> 用上面的多 Schema 配置，`graph.invoke({"question": "天气如何"})` 的返回值里会有哪些字段？
> 
> A. question、answer、context、reasoning 全部都有  
> B. 只有 answer  
> C. question 和 answer
> 
> 答案：**B**。`output_schema=OutputState` 决定了 invoke 的返回值只包含 `answer`。内部字段都被过滤掉了。

* * *

### 七、动手跑一下

**Demo 1：并行搜索 + Reducer**

下面用一个「并行搜索」的例子，演示 Reducer 的作用。

In [10]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END


# ---- 第1步：定义 State（带 Reducer）----
class SearchState(TypedDict):
    query: str
    results: Annotated[list, add]  # 关键：用 add 作为 Reducer


# ---- 第2步：定义节点 ----
def search_web(state: SearchState):
    print(f"[网页搜索] 搜索: {state['query']}")
    return {"results": ["网页结果1", "网页结果2"]}


def search_kb(state: SearchState):
    print(f"[知识库搜索] 搜索: {state['query']}")
    return {"results": ["知识库结果1"]}


def summarize(state: SearchState):
    print(f"[汇总] 收到 {len(state['results'])} 条结果")
    return {}  # 不修改任何字段


# ---- 第3步：组装图（并行执行）----
builder = StateGraph(SearchState)
builder.add_node("搜索网页", search_web)
builder.add_node("搜索知识库", search_kb)
builder.add_node("汇总", summarize)

# 从 START 同时出发到两个搜索节点
builder.add_edge(START, "搜索网页")
builder.add_edge(START, "搜索知识库")
# 两个搜索节点都完成后，到汇总节点
builder.add_edge("搜索网页", "汇总")
builder.add_edge("搜索知识库", "汇总")
builder.add_edge("汇总", END)

graph = builder.compile()

# ---- 第4步：运行 ----
result = graph.invoke({"query": "LangGraph 是什么", "results": []})
print(f"\n最终结果: {result}")

[知识库搜索] 搜索: LangGraph 是什么
[网页搜索] 搜索: LangGraph 是什么
[汇总] 收到 3 条结果

最终结果: {'query': 'LangGraph 是什么', 'results': ['知识库结果1', '网页结果1', '网页结果2']}


运行后你应该看到：

```text
[网页搜索] 搜索: LangGraph 是什么
[知识库搜索] 搜索: LangGraph 是什么
[汇总] 收到 3 条结果

最终结果: {'query': 'LangGraph 是什么', 'results': ['网页结果1', '网页结果2', '知识库结果1']}
```

注意看：`results` 里有 3 条结果，来自两个并行节点。这就是 Reducer 的作用！

**试试去掉 Reducer**

把 `Annotated[list, add]` 改成 `list`，再运行一次，看看会发生什么。

> 💡 **先猜一猜，再点开看答案：** 你会看到 `InvalidUpdateError: At key 'results': Can receive only one value per step.` 因为两个节点在同一步同时写 `results`，没有 Reducer，LangGraph 不知道该用哪个，所以直接报错——这比悄悄丢数据好得多。

**Demo 2：多轮对话 + 自定义字段**

这个例子演示两件事：

-   **继承 MessagesState** 添加自定义字段（`user_name`）
-   **多轮对话**：第二轮把第一轮的历史传回去，AI 能"记住"之前说过什么


In [11]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import HumanMessage, AIMessage


# ---- 第1步：继承 MessagesState，加自定义字段 ----
class ChatState(MessagesState):
    user_name: str  # messages 字段是继承来的，自带 add_messages


# ---- 第2步：定义节点 ----
def reply(state: ChatState):
    user_msg = state["messages"][-1].content  # 最新一条消息
    n = len(state["messages"])  # 当前历史长度
    text = f"{state['user_name']}，你说的是「{user_msg}」。这是我们对话的第 {n + 1} 条消息。"
    # 只返回【新增】的那条消息，add_messages 自动追加
    return {"messages": [AIMessage(content=text)]}


# ---- 第3步：组装图 ----
builder = StateGraph(ChatState)
builder.add_node("回复", reply)
builder.add_edge(START, "回复")
builder.add_edge("回复", END)

graph = builder.compile()

# ---- 第4步：第一轮对话 ----
state1 = graph.invoke({
    "messages": [HumanMessage(content="你好！")],
    "user_name": "小明",
})
print(f"第一轮后共 {len(state1['messages'])} 条消息")

# ---- 第5步：第二轮对话（带上第一轮的历史）----
state2 = graph.invoke({
    "messages": state1["messages"] + [HumanMessage(content="我的订单到哪了？")],
    "user_name": "小明",
})
print(f"第二轮后共 {len(state2['messages'])} 条消息\n")

for msg in state2["messages"]:
    role = "用户" if isinstance(msg, HumanMessage) else "AI"
    print(f"  {role}: {msg.content}")

第一轮后共 2 条消息
第二轮后共 4 条消息

  用户: 你好！
  AI: 小明，你说的是「你好！」。这是我们对话的第 2 条消息。
  用户: 我的订单到哪了？
  AI: 小明，你说的是「我的订单到哪了？」。这是我们对话的第 4 条消息。


运行后你应该看到：

```text
第一轮后共 2 条消息
第二轮后共 4 条消息

  用户: 你好！
  AI: 小明，你说的是「你好！」。这是我们对话的第 2 条消息。
  用户: 我的订单到哪了？
  AI: 小明，你说的是「我的订单到哪了？」。这是我们对话的第 4 条消息。
```

**关键观察：**

-   节点每次只返回**新增的 1 条** AI 消息，`add_messages` 自动追加，历史一条没丢
-   AI 能说出「第 4 条消息」，说明它**看得到完整历史**——这就是多轮记忆的本质
-   `user_name` 是普通字段（覆盖语义），和带 Reducer 的 `messages` 在同一个 State 里和平共处
-   真实聊天机器人就是把「生成回复」换成调用 LLM，循环往复——结构完全一样

> 💡 **为什么第二轮要手动拼 `state1["messages"] + [...]`？** 因为每次 `invoke` 都是全新的一次运行，图本身不保存上一次的 State。让图「自动记住」历史需要 **Checkpointer（持久化）**，那是后面课程的内容。

**Demo 3：单独把玩 add_messages**

`add_messages` 就是个普通函数：`add_messages(旧列表, 新列表) → 合并后的列表`。不用搭图，直接调用它，就能把「追加 / 替换 / 删除」三种行为看得清清楚楚。

In [12]:
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, RemoveMessage


# ① 追加：从空列表开始，加 2 条消息
history = add_messages(
    [],
    [
        HumanMessage(content="你好", id="h1"),
        AIMessage(content="你好！有什么可以帮你？", id="a1"),
    ],
)
print(f"① 追加 2 条 → {[m.content for m in history]}")

# ② 继续追加：新 id → 加到末尾
history = add_messages(history, [HumanMessage(content="订单到哪了？", id="h2")])
print(f"② 再追加 1 条 → {[m.content for m in history]}")

# ③ 替换：id="a1" 已存在 → 原地替换，不是追加！
history = add_messages(history, [AIMessage(content="您好呀！请问有什么问题？", id="a1")])
print(f"③ id=a1 重写 → {[m.content for m in history]}")

# ④ 删除：RemoveMessage 按 id 删掉对应消息
history = add_messages(history, [RemoveMessage(id="h1")])
print(f"④ 删除 id=h1 → {[m.content for m in history]}")

① 追加 2 条 → ['你好', '你好！有什么可以帮你？']
② 再追加 1 条 → ['你好', '你好！有什么可以帮你？', '订单到哪了？']
③ id=a1 重写 → ['你好', '您好呀！请问有什么问题？', '订单到哪了？']
④ 删除 id=h1 → ['您好呀！请问有什么问题？', '订单到哪了？']


运行后你应该看到：

```text
① 追加 2 条 → ['你好', '你好！有什么可以帮你？']
② 再追加 1 条 → ['你好', '你好！有什么可以帮你？', '订单到哪了？']
③ id=a1 重写 → ['你好', '您好呀！请问有什么问题？', '订单到哪了？']
④ 删除 id=h1 → ['您好呀！请问有什么问题？', '订单到哪了？']
```

**关键观察：**

-   步骤 ③：消息总数**没变**，第 2 条被原地替换——位置不变，内容更新
-   步骤 ④：`RemoveMessage` 本身不会出现在列表里，它只是个「删除指令」
-   在图里，节点返回的 `{"messages": [...]}` 就是这里的「新列表」参数，LangGraph 在背后帮你调用这个函数

> **检查理解 ④**
> 
> 当前 `messages` 里有 3 条消息（id 分别为 m1、m2、m3）。节点返回 `{"messages": [AIMessage(content="新内容", id="m2")]}`，执行后共有几条消息？
> 
> A. 4 条，新消息追加到末尾  
> B. 3 条，m2 的内容被替换  
> C. 1 条，旧消息全被覆盖
> 
> 答案：**B**。id=m2 已存在，`add_messages` 会原地替换它的内容，总数不变。这正是 Demo 3 步骤 ③ 演示的行为。

* * *

### 八、常见坑速查

| 症状  | 原因  | 解法  |
| --- | --- | --- |
| InvalidUpdateError: Can receive only one value per step | 并行节点写了同一个没有 Reducer 的字段 | 给字段加 Annotated\[list, add\] 或其他 Reducer |
| 列表里出现一堆单个字符 | 用了 add 但节点返回了字符串而非列表 | 返回值包成列表：{"results": \["xx"\]} |
| 对话历史只剩最后一条 | messages 字段没加 Reducer，被覆盖了 | 用 add_messages 或直接用 MessagesState |
| 消息越积越多、重复出现 | 把读到的整段历史又原样返回了一遍（无 id 视为新消息追加） | 只返回新增的消息；要替换就带相同 id |
| 继承 MessagesState 后追加失效 | 子类里又写了 messages: list，把 Reducer 覆盖了 | 继承后只添加新字段，别重新定义 messages |

* * *

### 九、总结

| 概念  | 一句话解释 | 类比  |
| --- | --- | --- |
| 覆盖（默认） | 新值替换旧值 | 快递站只留最新包裹 |
| Reducer | 定义多个新值如何合并：f(旧值, 新值) → 合并值 | 分拣规则 |
| operator.add | 列表追加 | 把包裹堆在一起 |
| add_messages | 智能消息合并（按 ID 更新/删除） | 智能记录本 |
| MessagesState | 内置的消息 State 快捷方式 | 现成的订单模板 |
| Annotated | 给字段附加 Reducer 信息 | 在字段上贴规则标签 |
| 多 Schema | 输入输出分离，隐藏内部细节 | 用户只看到订单和结果 |

> **核心记忆**  
> **默认 = 覆盖，Reducer = 合并规则**  
> 多个节点**同一步**写同一个字段时，必须用 Reducer。最常用的是 `operator.add`（列表追加）和 `add_messages`（消息管理）。

> **下节课预告**  
> State 会流转了，但目前所有图都是「一条道走到黑」。下一课学**条件路由**：让图根据 State 的内容，自己决定走哪条边。

* * *

📖 **参考：** [LangGraph State 官方文档](https://docs.langchain.com/oss/python/langgraph/graph-api#state) · [Reducers 文档](https://docs.langchain.com/oss/python/langgraph/graph-api#reducers)
